# Retail P4 Data Warehouse
End-to-end retail analytics pipeline: warehouse setup, data loading, and analytical queries.

In [ ]:
%%sql -r phase1_wh
-- PHASE 1: CREATE WAREHOUSE
CREATE WAREHOUSE IF NOT EXISTS RETAIL_P4_WH
WITH
WAREHOUSE_SIZE = 'XSMALL'
AUTO_SUSPEND = 60
AUTO_RESUME = TRUE;

In [ ]:
%%sql -r use_wh
USE WAREHOUSE RETAIL_P4_WH;

In [ ]:
%%sql -r phase2_db
-- PHASE 2: CREATE DATABASE AND SCHEMA
CREATE DATABASE IF NOT EXISTS RETAIL_P4_DB;

In [ ]:
%%sql -r phase2_schema
USE DATABASE RETAIL_P4_DB;
CREATE SCHEMA IF NOT EXISTS RETAIL_P4_SCHEMA;
USE SCHEMA RETAIL_P4_SCHEMA;

In [ ]:
%%sql -r phase3_ff
-- PHASE 3: CREATE FILE FORMAT
CREATE OR REPLACE FILE FORMAT RETAIL_P4_CSV_FORMAT
TYPE = 'CSV'
FIELD_DELIMITER = ','
SKIP_HEADER = 1
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
NULL_IF = ('NULL', 'null', '');

In [ ]:
%%sql -r phase4_stages
-- PHASE 4: CREATE STAGES
CREATE STAGE IF NOT EXISTS RETAIL_P4_CUSTOMER_STAGE FILE_FORMAT = RETAIL_P4_CSV_FORMAT;
CREATE STAGE IF NOT EXISTS RETAIL_P4_PRODUCT_STAGE FILE_FORMAT = RETAIL_P4_CSV_FORMAT;
CREATE STAGE IF NOT EXISTS RETAIL_P4_BRANCH_STAGE FILE_FORMAT = RETAIL_P4_CSV_FORMAT;
CREATE STAGE IF NOT EXISTS RETAIL_P4_CALENDAR_STAGE FILE_FORMAT = RETAIL_P4_CSV_FORMAT;
CREATE STAGE IF NOT EXISTS RETAIL_P4_SALES_STAGE FILE_FORMAT = RETAIL_P4_CSV_FORMAT;

In [ ]:
%%sql -r phase5_list
-- PHASE 5: VERIFY STAGES
LIST @RETAIL_P4_SALES_STAGE;

In [ ]:
%%sql -r phase6_customer
-- PHASE 6: CREATE DIMENSION TABLES
CREATE OR REPLACE TABLE DIM_CUSTOMER (
    CUSTOMER_ID INT PRIMARY KEY,
    CUSTOMER_NAME VARCHAR(100),
    CITY VARCHAR(50),
    STATE VARCHAR(50),
    MEMBERSHIP VARCHAR(20)
);

In [ ]:
%%sql -r phase6_product
CREATE OR REPLACE TABLE DIM_PRODUCT (
    PRODUCT_ID INT PRIMARY KEY,
    PRODUCT_NAME VARCHAR(100),
    CATEGORY VARCHAR(50),
    BRAND VARCHAR(50),
    PRICE DECIMAL(10,2)
);

In [ ]:
%%sql -r phase6_branch
CREATE OR REPLACE TABLE DIM_BRANCH (
    BRANCH_ID INT PRIMARY KEY,
    BRANCH_NAME VARCHAR(100),
    CITY VARCHAR(50),
    STATE VARCHAR(50),
    REGION VARCHAR(30),
    MANAGER_NAME VARCHAR(100)
);

In [ ]:
%%sql -r phase6_date
CREATE OR REPLACE TABLE DIM_DATE (
    DATE_ID INT PRIMARY KEY,
    DATE_VALUE DATE,
    DAY INT,
    DAY_NAME VARCHAR(20),
    WEEK_NO INT,
    MONTH VARCHAR(20),
    QUARTER VARCHAR(10),
    YEAR INT,
    IS_WEEKEND VARCHAR(5)
);

In [ ]:
%%sql -r phase7_fact
-- PHASE 7: CREATE FACT TABLE
CREATE OR REPLACE TABLE FACT_SALES (
    SALE_ID INT PRIMARY KEY,
    CUSTOMER_ID INT,
    PRODUCT_ID INT,
    BRANCH_ID INT,
    DATE_ID INT,
    QUANTITY INT,
    TOTAL_AMOUNT DECIMAL(15,2),
    FOREIGN KEY (CUSTOMER_ID) REFERENCES DIM_CUSTOMER(CUSTOMER_ID),
    FOREIGN KEY (PRODUCT_ID) REFERENCES DIM_PRODUCT(PRODUCT_ID),
    FOREIGN KEY (BRANCH_ID) REFERENCES DIM_BRANCH(BRANCH_ID),
    FOREIGN KEY (DATE_ID) REFERENCES DIM_DATE(DATE_ID)
);

In [ ]:
%%sql -r phase8_load
-- PHASE 8: LOAD CUSTOMER CSV
COPY INTO DIM_CUSTOMER
FROM @RETAIL_P4_SALES_STAGE/customers.csv
FILE_FORMAT = RETAIL_P4_CSV_FORMAT
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r phase9_load
-- PHASE 9: LOAD PRODUCT CSV
COPY INTO DIM_PRODUCT
FROM @RETAIL_P4_SALES_STAGE/products.csv
FILE_FORMAT = RETAIL_P4_CSV_FORMAT
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r phase10_load
-- PHASE 10: LOAD BRANCH CSV
COPY INTO DIM_BRANCH
FROM @RETAIL_P4_SALES_STAGE/branches.csv
FILE_FORMAT = RETAIL_P4_CSV_FORMAT
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r phase11_load
-- PHASE 11: LOAD CALENDAR CSV
COPY INTO DIM_DATE
FROM @RETAIL_P4_SALES_STAGE/calendar.csv
FILE_FORMAT = RETAIL_P4_CSV_FORMAT
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r phase12_load
-- PHASE 12: LOAD SALES CSV
COPY INTO FACT_SALES
FROM @RETAIL_P4_SALES_STAGE/sales.csv
FILE_FORMAT = RETAIL_P4_CSV_FORMAT
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r phase13_verify
-- PHASE 13: VERIFY DATA
SELECT 'DIM_CUSTOMER' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM DIM_CUSTOMER
UNION ALL SELECT 'DIM_PRODUCT', COUNT(*) FROM DIM_PRODUCT
UNION ALL SELECT 'DIM_BRANCH', COUNT(*) FROM DIM_BRANCH
UNION ALL SELECT 'DIM_DATE', COUNT(*) FROM DIM_DATE
UNION ALL SELECT 'FACT_SALES', COUNT(*) FROM FACT_SALES;

In [ ]:
%%sql -r phase14_cust_sales
-- PHASE 14: CUSTOMER-WISE SALES
SELECT
    C.CUSTOMER_ID,
    C.CUSTOMER_NAME,
    SUM(F.TOTAL_AMOUNT) AS TOTAL_SALES
FROM FACT_SALES F
JOIN DIM_CUSTOMER C ON F.CUSTOMER_ID = C.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID, C.CUSTOMER_NAME
ORDER BY TOTAL_SALES DESC;

In [ ]:
%%sql -r phase15_prod_rev
-- PHASE 15: PRODUCT-WISE REVENUE
SELECT
    P.PRODUCT_ID,
    P.PRODUCT_NAME,
    SUM(F.TOTAL_AMOUNT) AS REVENUE
FROM FACT_SALES F
JOIN DIM_PRODUCT P ON F.PRODUCT_ID = P.PRODUCT_ID
GROUP BY P.PRODUCT_ID, P.PRODUCT_NAME
ORDER BY REVENUE DESC;

In [ ]:
%%sql -r phase16_branch_sales
-- PHASE 16: BRANCH-WISE SALES
SELECT
    B.BRANCH_ID,
    B.BRANCH_NAME,
    SUM(F.TOTAL_AMOUNT) AS TOTAL_SALES
FROM FACT_SALES F
JOIN DIM_BRANCH B ON F.BRANCH_ID = B.BRANCH_ID
GROUP BY B.BRANCH_ID, B.BRANCH_NAME
ORDER BY TOTAL_SALES DESC;

In [ ]:
%%sql -r phase17_monthly
-- PHASE 17: MONTHLY REVENUE
SELECT
    D.YEAR,
    D.MONTH,
    SUM(F.TOTAL_AMOUNT) AS MONTHLY_REVENUE
FROM FACT_SALES F
JOIN DIM_DATE D ON F.DATE_ID = D.DATE_ID
GROUP BY D.YEAR, D.MONTH
ORDER BY D.YEAR, D.MONTH;

In [ ]:
%%sql -r phase18_state
-- PHASE 18: STATE-WISE REVENUE
SELECT
    C.STATE,
    SUM(F.TOTAL_AMOUNT) AS REVENUE
FROM FACT_SALES F
JOIN DIM_CUSTOMER C ON F.CUSTOMER_ID = C.CUSTOMER_ID
GROUP BY C.STATE
ORDER BY REVENUE DESC;

In [ ]:
%%sql -r phase19_category
-- PHASE 19: CATEGORY-WISE REVENUE
SELECT
    P.CATEGORY,
    SUM(F.TOTAL_AMOUNT) AS REVENUE
FROM FACT_SALES F
JOIN DIM_PRODUCT P ON F.PRODUCT_ID = P.PRODUCT_ID
GROUP BY P.CATEGORY
ORDER BY REVENUE DESC;

In [ ]:
%%sql -r phase20_top_cust
-- PHASE 20: TOP 10 CUSTOMERS
SELECT
    C.CUSTOMER_ID,
    C.CUSTOMER_NAME,
    SUM(F.TOTAL_AMOUNT) AS TOTAL_SALES
FROM FACT_SALES F
JOIN DIM_CUSTOMER C ON F.CUSTOMER_ID = C.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID, C.CUSTOMER_NAME
ORDER BY TOTAL_SALES DESC
LIMIT 10;

In [ ]:
%%sql -r phase21_top_prod
-- PHASE 21: TOP 10 PRODUCTS
SELECT
    P.PRODUCT_ID,
    P.PRODUCT_NAME,
    SUM(F.TOTAL_AMOUNT) AS REVENUE
FROM FACT_SALES F
JOIN DIM_PRODUCT P ON F.PRODUCT_ID = P.PRODUCT_ID
GROUP BY P.PRODUCT_ID, P.PRODUCT_NAME
ORDER BY REVENUE DESC
LIMIT 10;

In [ ]:
%%sql -r phase22_top_branch
-- PHASE 22: TOP 10 BRANCHES
SELECT
    B.BRANCH_ID,
    B.BRANCH_NAME,
    SUM(F.TOTAL_AMOUNT) AS REVENUE
FROM FACT_SALES F
JOIN DIM_BRANCH B ON F.BRANCH_ID = B.BRANCH_ID
GROUP BY B.BRANCH_ID, B.BRANCH_NAME
ORDER BY REVENUE DESC
LIMIT 10;

In [ ]:
%%sql -r phase23_trend
-- PHASE 23: SALES TREND
SELECT
    D.DATE_VALUE,
    SUM(F.TOTAL_AMOUNT) AS DAILY_SALES
FROM FACT_SALES F
JOIN DIM_DATE D ON F.DATE_ID = D.DATE_ID
GROUP BY D.DATE_VALUE
ORDER BY D.DATE_VALUE;

In [ ]:
%%sql -r phase24_quarterly
-- PHASE 24: QUARTERLY REVENUE
SELECT
    D.YEAR,
    D.QUARTER,
    SUM(F.TOTAL_AMOUNT) AS REVENUE
FROM FACT_SALES F
JOIN DIM_DATE D ON F.DATE_ID = D.DATE_ID
GROUP BY D.YEAR, D.QUARTER
ORDER BY D.YEAR, D.QUARTER;

In [ ]:
%%sql -r phase25_purchase
-- PHASE 25: CUSTOMER PURCHASE ANALYSIS
SELECT
    C.CUSTOMER_NAME,
    C.MEMBERSHIP,
    COUNT(F.SALE_ID) AS NUMBER_OF_PURCHASES,
    SUM(F.QUANTITY) AS TOTAL_QUANTITY,
    SUM(F.TOTAL_AMOUNT) AS TOTAL_SPENDING
FROM FACT_SALES F
JOIN DIM_CUSTOMER C ON F.CUSTOMER_ID = C.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID, C.CUSTOMER_NAME, C.MEMBERSHIP
ORDER BY TOTAL_SPENDING DESC;

In [ ]:
%%sql -r phase26_product_perf
-- PHASE 26: PRODUCT PERFORMANCE
SELECT
    P.PRODUCT_NAME,
    P.CATEGORY,
    P.BRAND,
    SUM(F.QUANTITY) AS TOTAL_QUANTITY_SOLD,
    SUM(F.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES F
JOIN DIM_PRODUCT P ON F.PRODUCT_ID = P.PRODUCT_ID
GROUP BY P.PRODUCT_ID, P.PRODUCT_NAME, P.CATEGORY, P.BRAND
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r phase27_branch_perf
-- PHASE 27: BRANCH PERFORMANCE
SELECT
    B.BRANCH_NAME,
    B.CITY,
    B.STATE,
    B.REGION,
    SUM(F.QUANTITY) AS TOTAL_QUANTITY,
    SUM(F.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES F
JOIN DIM_BRANCH B ON F.BRANCH_ID = B.BRANCH_ID
GROUP BY B.BRANCH_ID, B.BRANCH_NAME, B.CITY, B.STATE, B.REGION
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r phase28_regional
-- PHASE 28: REGIONAL SALES
SELECT
    B.REGION,
    SUM(F.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES F
JOIN DIM_BRANCH B ON F.BRANCH_ID = B.BRANCH_ID
GROUP BY B.REGION
ORDER BY TOTAL_REVENUE DESC;